In [ ]:
from google.colab import drive
import os

# This will prompt you to log in and grant access
drive.mount('/content/drive')

# Create a dedicated folder for the hackathon
work_dir = '/content/drive/MyDrive/Redrob_Hackathon'
os.makedirs(work_dir, exist_ok=True)
print(f"Working directory ready at: {work_dir}")

Mounted at /content/drive
Working directory ready at: /content/drive/MyDrive/Redrob_Hackathon


In [ ]:
import os
work_dir = '/content/drive/MyDrive/Redrob_Hackathon'

if os.path.exists(work_dir):
    print("Files currently in your hackathon folder:")
    print(os.listdir(work_dir))
else:
    print("Colab cannot find the Redrob_Hackathon folder at all!")

Files currently in your hackathon folder:
['candidates.jsonl']


In [ ]:
import json

# FIX: Removed the .gz from the end of the filename here!
data_path = f"{work_dir}/candidates.jsonl"

def stream_candidates(filepath, max_records=None):
    """Yields candidates one by one to save RAM."""
    with open(filepath, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_records and i >= max_records:
                break
            if line.strip():
                yield json.loads(line)

# Test it by printing just the first candidate
candidate_generator = stream_candidates(data_path, max_records=1)
print(next(candidate_generator)['candidate_id'])

CAND_0000001


In [ ]:
import pandas as pd
import re
import os

# We compile a regex pattern of the explicitly banned service firms
service_firms = re.compile(r'\b(tcs|infosys|wipro|accenture|cognizant|capgemini|tech mahindra|hcl)\b', re.IGNORECASE)

def create_training_data(filepath, chunk_size=10000):
    records = []
    chunk_id = 0

    print("Starting label generation. This might take a few minutes...")

    for i, cand in enumerate(stream_candidates(filepath)):
        history = cand.get('career_history', [])

        # 1. Combine text for the BERT model to read
        # We take the headline, summary, and all job descriptions
        profile = cand.get('profile', {})
        full_text = f"{profile.get('headline', '')} {profile.get('summary', '')} "

        service_job_count = 0
        total_jobs = len(history)

        for job in history:
            company = str(job.get('company', ''))
            full_text += f"{job.get('description', '')} "

            # Check if the company matches our banned list
            if service_firms.search(company):
                service_job_count += 1

        # 2. Labeling Logic
        # If they have jobs, and EVERY job was at a service firm, label 0. Otherwise, label 1.
        if total_jobs > 0 and service_job_count == total_jobs:
            label = 0  # Pure Service (Reject)
        else:
            label = 1  # Product / Mix (Accept)

        # We truncate text to 1000 characters so our CSVs don't get too massive
        records.append({
            'candidate_id': cand.get('candidate_id'),
            'text': full_text[:1000].strip(),
            'label': label
        })

        # 3. Save in chunks to prevent data loss if Colab crashes
        if len(records) >= chunk_size:
            df = pd.DataFrame(records)
            save_path = f"{work_dir}/training_data_part_{chunk_id}.csv"
            df.to_csv(save_path, index=False)
            records = []
            chunk_id += 1
            print(f"Saved chunk {chunk_id} (Processed {(chunk_id * chunk_size):,} candidates)")

    # Save any remaining records
    if records:
        df = pd.DataFrame(records)
        df.to_csv(f"{work_dir}/training_data_part_{chunk_id}.csv", index=False)
        print(f"Saved final chunk {chunk_id}. All 100,000 processed!")

# Run the labeler
create_training_data(data_path)

Starting label generation. This might take a few minutes...
Saved chunk 1 (Processed 10,000 candidates)
Saved chunk 2 (Processed 20,000 candidates)
Saved chunk 3 (Processed 30,000 candidates)
Saved chunk 4 (Processed 40,000 candidates)
Saved chunk 5 (Processed 50,000 candidates)
Saved chunk 6 (Processed 60,000 candidates)
Saved chunk 7 (Processed 70,000 candidates)
Saved chunk 8 (Processed 80,000 candidates)
Saved chunk 9 (Processed 90,000 candidates)
Saved chunk 10 (Processed 100,000 candidates)


In [ ]:
# 1. Re-mount Google Drive so Colab can see your files again
from google.colab import drive
import os

drive.mount('/content/drive')

# 2. Re-define the work_dir variable
work_dir = '/content/drive/MyDrive/Redrob_Hackathon'
print(f"Directory set to: {work_dir}")

# 3. Quick check to make sure it sees your training CSVs and checkpoints
print("\nFiles found:")
print(os.listdir(work_dir))

Mounted at /content/drive
Directory set to: /content/drive/MyDrive/Redrob_Hackathon

Files found:
['candidates.jsonl', 'training_data_part_0.csv', 'training_data_part_1.csv', 'training_data_part_2.csv', 'training_data_part_3.csv', 'training_data_part_4.csv', 'training_data_part_5.csv', 'training_data_part_6.csv', 'training_data_part_7.csv', 'training_data_part_8.csv', 'training_data_part_9.csv', 'consultant_bert_checkpoints']


In [ ]:
# 1. Install required ML libraries
!pip install -q transformers datasets accelerate torch

import glob
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch

# 2. Combine our CSV chunks into a HuggingFace Dataset
print("Loading data...")
all_csvs = glob.glob(f"{work_dir}/training_data_part_*.csv")
dataset = load_dataset("csv", data_files=all_csvs, split="train")

# Shuffle and split: 90% for training, 10% for testing
dataset = dataset.shuffle(seed=42).train_test_split(test_size=0.1)

# 3. Load the Tokenizer and Model
print("Downloading DistilBERT...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# We have 2 labels: 0 (Reject) and 1 (Accept)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 4. Tokenize the text (convert words to numbers the AI understands)
print("Tokenizing text...")
def tokenize_function(examples):
    # We cap length at 128 tokens to keep training fast and prevent RAM crashes
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 5. Set up Training with Checkpointing
training_args = TrainingArguments(
    output_dir=f"{work_dir}/consultant_bert_checkpoints",
    eval_strategy="epoch",            # <--- FIXED THIS LINE!
    save_strategy="steps",            # Save progress periodically
    save_steps=1000,                  # Save a checkpoint every 1000 batches
    per_device_train_batch_size=16,   # Small batch size to fit in Colab's 12GB RAM
    per_device_eval_batch_size=16,
    num_train_epochs=1,               # 1 epoch is enough for a hackathon baseline
    fp16=torch.cuda.is_available(),   # Use GPU acceleration if Colab gave you one
    resume_from_checkpoint=True,      # Automatically resume if it crashed previously
    report_to="none"                  # Disable external logging
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

# 6. Start Training!
print("Starting training...")
trainer.train()

# 7. Save the final completed model to your Google Drive
print("Training complete! Saving final model...")
model.save_pretrained(f"{work_dir}/final_consultant_bert")
tokenizer.save_pretrained(f"{work_dir}/final_consultant_bert")
print("Model saved successfully!")

Loading data...


Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# 1. Install required ML libraries
!pip install -q transformers datasets accelerate torch

import glob
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch

# 2. Combine our CSV chunks into a HuggingFace Dataset
print("Loading data...")
all_csvs = glob.glob(f"{work_dir}/training_data_part_*.csv")
dataset = load_dataset("csv", data_files=all_csvs, split="train")

# Shuffle and split: 90% for training, 10% for testing
dataset = dataset.shuffle(seed=42).train_test_split(test_size=0.1)

# 3. Load the Tokenizer and Model
print("Downloading DistilBERT...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# We have 2 labels: 0 (Reject) and 1 (Accept)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 4. Tokenize the text (convert words to numbers the AI understands)
print("Tokenizing text...")
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 5. Set up Training with Checkpointing
training_args = TrainingArguments(
    output_dir=f"{work_dir}/consultant_bert_checkpoints",
    eval_strategy="epoch",
    save_strategy="steps",            # Save progress periodically
    save_steps=1000,                  # Save a checkpoint every 1000 batches
    per_device_train_batch_size=16,   # Small batch size to fit in Colab's 12GB RAM
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),   # Use GPU acceleration if Colab gave you one
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

# 6. Start Training and Resume from Checkpoint!
print("Resuming training from last saved checkpoint...")
# FIX: Moved resume_from_checkpoint here directly into the train function!
trainer.train(resume_from_checkpoint=True)

# 7. Save the final completed model to your Google Drive
print("Training complete! Saving final model...")
model.save_pretrained(f"{work_dir}/final_consultant_bert")
tokenizer.save_pretrained(f"{work_dir}/final_consultant_bert")
print("Model saved successfully!")

Loading data...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizing text...


Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training complete! Saving final model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [ ]:
# 1. Install required libraries
!pip install -q faiss-cpu sentence-transformers

# ---> Explicitly mount Drive first to cure the amnesia! <---
from google.colab import drive
import os
drive.mount('/content/drive')

import faiss
import numpy as np
import json
from sentence_transformers import SentenceTransformer

# 2. Define paths and data streamer
work_dir = '/content/drive/MyDrive/Redrob_Hackathon'
data_path = f"{work_dir}/candidates.jsonl"

def stream_candidates(filepath):
    with open(filepath, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

# 3. Load the embedding model (Fast and lightweight)
print("Loading Sentence Transformer model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
vector_dimension = 384

# 4. Initialize the FAISS index (Inner Product for cosine similarity)
index = faiss.IndexFlatIP(vector_dimension)

candidate_ids = []
batch_texts = []
batch_ids = []
total_indexed = 0

print("Generating embeddings and building FAISS index. This will take some time...")

for cand in stream_candidates(data_path):
    # We embed the headline and summary for the initial semantic match
    profile = cand.get('profile', {})
    text = f"{profile.get('headline', '')} {profile.get('summary', '')}"

    batch_texts.append(text)
    batch_ids.append(cand.get('candidate_id'))

    # Process in batches of 2,000 to keep RAM safe
    if len(batch_texts) == 2000:
        embeddings = embedder.encode(batch_texts, convert_to_numpy=True)
        faiss.normalize_L2(embeddings) # Normalize so Inner Product = Cosine Similarity
        index.add(embeddings)
        candidate_ids.extend(batch_ids)

        total_indexed += 2000
        print(f"Indexed {total_indexed} candidates...")

        # Clear batches
        batch_texts, batch_ids = [], []

# 5. Process the final leftover batch
if batch_texts:
    embeddings = embedder.encode(batch_texts, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)
    index.add(embeddings)
    candidate_ids.extend(batch_ids)
    total_indexed += len(batch_texts)
    print(f"Indexed final batch. Total: {total_indexed} candidates.")

# 6. Save the Index and the ID mapping to Google Drive
print("Saving FAISS index and ID mappings to Google Drive...")
faiss.write_index(index, f"{work_dir}/candidates.index")
with open(f"{work_dir}/id_mapping.json", "w") as f:
    json.dump(candidate_ids, f)

print("Phase 4 Complete! Index saved successfully.")

Mounted at /content/drive
Loading Sentence Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings and building FAISS index. This will take some time...
Indexed 2000 candidates...
Indexed 4000 candidates...
Indexed 6000 candidates...
Indexed 8000 candidates...
Indexed 10000 candidates...
Indexed 12000 candidates...
Indexed 14000 candidates...
Indexed 16000 candidates...
Indexed 18000 candidates...
Indexed 20000 candidates...
Indexed 22000 candidates...
Indexed 24000 candidates...
Indexed 26000 candidates...
Indexed 28000 candidates...
Indexed 30000 candidates...
Indexed 32000 candidates...
Indexed 34000 candidates...
Indexed 36000 candidates...
Indexed 38000 candidates...
Indexed 40000 candidates...
Indexed 42000 candidates...
Indexed 44000 candidates...
Indexed 46000 candidates...
Indexed 48000 candidates...
Indexed 50000 candidates...
Indexed 52000 candidates...
Indexed 54000 candidates...
Indexed 56000 candidates...
Indexed 58000 candidates...
Indexed 60000 candidates...
Indexed 62000 candidates...
Indexed 64000 candidates...
Indexed 66000 candidates...
Inde